# E91 Quantum Key Distribution

This notebook demonstrates the E91 Quantum Key Distribution protocol from Chapter 10, Key Applications of Entanglement.

[![Open In Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Polaris-QCI/Quantum-Computing-and-Information-ed2/main?labpath=Qiskit%2F10-Key-Applications-of-Entanglement%2Fe91-quantum-key-distribution-demo.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Polaris-QCI/Quantum-Computing-and-Information-ed2/blob/main/Qiskit/10-Key-Applications-of-Entanglement/e91-quantum-key-distribution-demo.ipynb)

## 0. Setup

The cells below install the packages and import the libraries needed by this demo.

In [1]:
#@title Install Qiskit and other dependencies
%pip install pylatexenc qiskit qiskit-aer qiskit-ibm-runtime

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 18.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 26.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.8 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.7 MB/s  0:00:00
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136897 sha256=d62ec45108cce7456f4ca2787f889e5154634aaa3692f4018fb2336e50bf3225
  Stored in directory: /Users/jakeschaefer/Library/Caches/

In [ ]:
# Imports

import random
from IPython.display import display, Markdown
from math import cos, sin, pi as π
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit import ClassicalRegister, QuantumCircuit, QuantumRegister



<class 'type'>


## 1. E91 bit

The next cell implements the E91 protocol for a single potential key bit. 

In [ ]:
def e91_bit():
  q = [q0, q1] = QuantumRegister(2)
  c = ClassicalRegister(2, 'c')
  circuit = QuantumCircuit(q, c)

  # Generate the Ψ⁻ Bell state
  circuit.x(q0)
  circuit.x(q1)
  circuit.h(q0)
  circuit.cx(q0, q1)

  # Alice's basis choices
  # a1 = Z (no rotation needed)
  # a2 = W = 1/√2(Z+X)
  # a3 = X
  rand = random.randint(0, 2)
  if rand == 0:  # Z basis
      alice_basis = "a1"
  if rand == 1:  # W basis
      alice_basis = "a2"
      circuit.ry(-π/4, q0)
  elif rand == 2:  # X basis
      alice_basis = "a3"
      circuit.h(q0)

  # Bob's basis choices
  # b1 = W = 1/√2(Z+X)
  # b2 = X
  # b3 = V = 1/√2(Z-X)
  rand = random.randint(0, 2)
  if rand == 0:  # W basis
      bob_basis = "b1"
      circuit.ry(-π/4, q1)
  elif rand == 1:  # X basis
      bob_basis = "b2"
      circuit.h(q1)
  else:  # V basis
      bob_basis = "b3"
      circuit.ry(π/4, q1)

  circuit.measure(q, c)

  simulator = AerSimulator()
  sampler = Sampler(simulator)
  job = sampler.run([circuit], shots=1)
  job_result = job.result()
  result = next(iter(job_result[0].data.c.get_counts().keys()))
  alice_result = result[0]
  bob_result = result[1]
  return {
    "alice_basis": alice_basis,
    "bob_basis": bob_basis,
    "alice_result": alice_result,
    "bob_result": bob_result,
  }


In [ ]:
output =  "| # | Alice Basis | Bob Basis | Use for Key | Measurement Result | Use for Checking |\n"
output += "|---|-------------|-----------|-------------|----------------|--------------|\n"
for i in range(20):
  bit = e91_bit()
  alice_basis = bit['alice_basis']
  bob_basis = bit['bob_basis']
  use_for_key = "✓" if alice_basis == "a2" and bob_basis == "b1" or alice_basis == "a3" and bob_basis == "b2" else " "
  measurement_result = f"**{bit['alice_result']}**" if use_for_key == "✓" else bit['alice_result']
  use_for_checking = "✓" if alice_basis != "a2" and bob_basis != "b2" else "✗"
  output += f"| {i+1} | {alice_basis} | {bob_basis} | {use_for_key} | {measurement_result} | {use_for_checking} |\n"
display(Markdown(output))

<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>
<class 'qiskit.circuit.ClassicalRegister'>


| # | Alice Basis | Bob Basis | Use for Key | Measurement Result | Use for Checking |
|---|-------------|-----------|-------------|----------------|--------------|
| 1 | a3 | b3 |   | 0 | ✓ |
| 2 | a1 | b1 |   | 0 | ✓ |
| 3 | a2 | b2 |   | 0 | ✗ |
| 4 | a2 | b3 |   | 1 | ✗ |
| 5 | a2 | b2 |   | 1 | ✗ |
| 6 | a1 | b3 |   | 1 | ✓ |
| 7 | a3 | b1 |   | 0 | ✓ |
| 8 | a3 | b3 |   | 0 | ✓ |
| 9 | a2 | b2 |   | 1 | ✗ |
| 10 | a2 | b3 |   | 0 | ✗ |
| 11 | a3 | b1 |   | 1 | ✓ |
| 12 | a2 | b3 |   | 0 | ✗ |
| 13 | a2 | b2 |   | 0 | ✗ |
| 14 | a3 | b1 |   | 1 | ✓ |
| 15 | a3 | b1 |   | 0 | ✓ |
| 16 | a1 | b2 |   | 1 | ✗ |
| 17 | a2 | b3 |   | 0 | ✗ |
| 18 | a3 | b3 |   | 0 | ✓ |
| 19 | a1 | b2 |   | 0 | ✗ |
| 20 | a2 | b2 |   | 1 | ✗ |
